<a href="https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Scope note

This notebook mirrors `index.html` (the deployed paper) section by section, recomputing
the key numbers from `work/model_df_export.csv` and `work/baseline_action_score.csv` so
every figure below is verified in this run, not copy-pasted from earlier notebooks.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
print("Lane: Refresh / Content Opportunity Scoring")
print()
print("Research question: Which pages should be reviewed first for refresh, expansion,")
print("protection, pruning, or monitoring, based on their observed search and engagement")
print("signals?")
print()
print("Decision supported: allocate a content/search team's limited weekly review")
print("capacity to the pages most worth a human look this cycle.")
print()
print("Unit of analysis: one content page, identified by content_hash_id.")
print()
print("Output: a ranked review queue with a score and a reason code -- never an automatic")
print("refresh/expand/protect/prune/monitor decision.")
print()
print("Cost of a wrong recommendation: a false positive wastes limited review time; a")
print("false negative lets a genuinely declining page go unreviewed another cycle.")


Lane: Refresh / Content Opportunity Scoring

Research question: Which pages should be reviewed first for refresh, expansion,
protection, pruning, or monitoring, based on their observed search and engagement
signals?

Decision supported: allocate a content/search team's limited weekly review
capacity to the pages most worth a human look this cycle.

Unit of analysis: one content page, identified by content_hash_id.

Output: a ranked review queue with a score and a reason code -- never an automatic
refresh/expand/protect/prune/monitor decision.

Cost of a wrong recommendation: a false positive wastes limited review time; a
false negative lets a genuinely declining page go unreviewed another cycle.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
import pandas as pd

model_df = pd.read_csv("work/model_df_export.csv")
baseline = pd.read_csv("work/baseline_action_score.csv")

print("Release: FlyRank ML Internship warehouse (Hugging Face)")
print("Window: March 2026 (features) with an April 2026 comparison for the label")
print("Full window volume: 9,841,378 rows across 55 anonymized clients (per w03_data_contract)")
print()
print("Unique content pages in the feature/label export:", model_df["content_hash_id"].nunique())
print("Unique content pages in the baseline export:      ", baseline["content_hash_id"].nunique())
print()
print("Excluded, and why:")
print("- march_clicks: part of the label's own definition (deliberate leakage trap, see Methodology)")
print("- the sealed _sample table: reserved outside feature development by the dataset's own contract")
print("- any client name, domain, URL, or private query: not present anywhere in these exports")


Release: FlyRank ML Internship warehouse (Hugging Face)
Window: March 2026 (features) with an April 2026 comparison for the label
Full window volume: 9,841,378 rows across 55 anonymized clients (per w03_data_contract)

Unique content pages in the feature/label export: 331437
Unique content pages in the baseline export:       331437

Excluded, and why:
- march_clicks: part of the label's own definition (deliberate leakage trap, see Methodology)
- the sealed _sample table: reserved outside feature development by the dataset's own contract
- any client name, domain, URL, or private query: not present anywhere in these exports


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
feature_cols = ["march_impressions", "avg_search_position", "march_sessions",
                "engagement_rate", "march_scroll_events"]

print("Baseline: hand-written rule on march_impressions + staleness_days,")
print("cut at baseline_score >= 70.0 ->", (baseline["baseline_score"] >= 70).sum(),
      "of", len(baseline), "pages flagged REVIEW_REFRESH")
print()
print("Label: is_declining_proxy = (april_clicks < march_clicks), built one step")
print("upstream from this feature set so the split below needs no extra time-awareness.")
print()
print("Final feature set (march_clicks excluded):", feature_cols)
print()
print("Leakage trap (from w03_data_contract): training WITH march_clicks included gave a")
print("leaked ROC AUC of 0.9677; removing it dropped that to an honest 0.9238.")
print()
print("Split: random, stratified 80/20 at the content-page level.")
print("Unique content_hash_id count == row count:",
      model_df["content_hash_id"].nunique() == len(model_df), "-> no page appears twice,")
print("so a grouped-by-client split isn't needed to prevent same-page leakage, and this")
print("export has no client_hash_id to group by regardless.")


Baseline: hand-written rule on march_impressions + staleness_days,
cut at baseline_score >= 70.0 -> 10189 of 331437 pages flagged REVIEW_REFRESH

Label: is_declining_proxy = (april_clicks < march_clicks), built one step
upstream from this feature set so the split below needs no extra time-awareness.

Final feature set (march_clicks excluded): ['march_impressions', 'avg_search_position', 'march_sessions', 'engagement_rate', 'march_scroll_events']

Leakage trap (from w03_data_contract): training WITH march_clicks included gave a
leaked ROC AUC of 0.9677; removing it dropped that to an honest 0.9238.

Split: random, stratified 80/20 at the content-page level.
Unique content_hash_id count == row count: True -> no page appears twice,
so a grouped-by-client split isn't needed to prevent same-page leakage, and this
export has no client_hash_id to group by regardless.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

df = model_df.merge(baseline[["content_hash_id", "baseline_score"]], on="content_hash_id", how="inner")
X = df[feature_cols].copy()
y = df["is_declining_proxy"].copy()

def precision_at_k(labels, scores, k=50):
    order = np.argsort(-scores)[:k]
    return labels.to_numpy()[order].mean()

def fit_and_score(train_idx, test_idx, seed=42):
    imputer = SimpleImputer(strategy="median")
    Xtr = imputer.fit_transform(X.loc[train_idx]); Xte = imputer.transform(X.loc[test_idx])
    ytr, yte = y.loc[train_idx], y.loc[test_idx]
    m = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                class_weight="balanced", random_state=seed, n_jobs=-1)
    m.fit(Xtr, ytr)
    return precision_at_k(yte, m.predict_proba(Xte)[:, 1], k=50), m

train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42, stratify=y)
model_p50, fitted_model = fit_and_score(train_idx, test_idx)
baseline_p50 = precision_at_k(y.loc[test_idx], df.loc[test_idx, "baseline_score"].to_numpy(), k=50)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = np.array([fit_and_score(df.index[tr], df.index[te])[0] for tr, te in skf.split(X, y)])

print("Precision@50, single 80/20 split:")
print(f"  Baseline (Week-4 rule): {baseline_p50:.2f}")
print(f"  Model (Random Forest):  {model_p50:.2f}")
print()
print(f"5-fold CV, model Precision@50: {np.round(fold_scores, 4)}")
print(f"  mean {fold_scores.mean():.3f}  std {fold_scores.std():.3f}")
print()
importances = pd.Series(fitted_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances.round(4))


Precision@50, single 80/20 split:
  Baseline (Week-4 rule): 0.42
  Model (Random Forest):  0.66

5-fold CV, model Precision@50: [0.7  0.8  0.78 0.7  0.66]
  mean 0.728  std 0.053

Feature importances:
march_impressions      0.6320
march_sessions         0.1709
avg_search_position    0.1695
march_scroll_events    0.0220
engagement_rate        0.0056
dtype: float64


## 5. Limitations

*What this work cannot claim.*

In [ ]:
zero_impr = df["march_impressions"] == 0
print("Structural artifact in the label:")
print("Rows with march_impressions == 0:", int(zero_impr.sum()))
print("Of those, is_declining_proxy == 1:", int((zero_impr & (df['is_declining_proxy'] == 1)).sum()))
print("-> mechanically impossible for a zero-traffic page to be labeled declining,")
print("   since april_clicks < march_clicks can't hold when march_clicks == 0.")
print()
staleness_rate = baseline["staleness_days"].notna().mean()
print(f"Staleness coverage gap: staleness_days is recorded for only {staleness_rate:.1%} of pages.")
print()
print("What this work does NOT claim:")
print("- Not causal: no signal is shown to CAUSE decline, only associated with the proxy.")
print("- Not validated out-of-time: all splits are within the same March-derived,")
print("  April-anchored development window; no later month was held out.")
print("- Not a verdict: is_declining_proxy is one proxy definition, not ground truth.")
print()
bold = "The model predicts which pages will decline in performance."
safe = ("Ranking by the model's predicted probability put more pages matching the observed "
        "April-vs-March decline proxy into the top 50 than the baseline did (0.66 vs 0.42, "
        "same test rows) -- an observed, directional improvement on one proxy label, not a "
        "forecast of future decline.")
print("Bold claim (avoided):", bold)
print("Rewritten claim (used):", safe)


Structural artifact in the label:
Rows with march_impressions == 0: 154699
Of those, is_declining_proxy == 1: 0
-> mechanically impossible for a zero-traffic page to be labeled declining,
   since april_clicks < march_clicks can't hold when march_clicks == 0.

Staleness coverage gap: staleness_days is recorded for only 11.5% of pages.

What this work does NOT claim:
- Not causal: no signal is shown to CAUSE decline, only associated with the proxy.
- Not validated out-of-time: all splits are within the same March-derived,
  April-anchored development window; no later month was held out.
- Not a verdict: is_declining_proxy is one proxy definition, not ground truth.

Bold claim (avoided): The model predicts which pages will decline in performance.
Rewritten claim (used): Ranking by the model's predicted probability put more pages matching the observed April-vs-March decline proxy into the top 50 than the baseline did (0.66 vs 0.42, same test rows) -- an observed, directional improvement o

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
queue = baseline[baseline["action"] == "REVIEW_REFRESH"].sort_values("baseline_score", ascending=False)
print("Review queue size:", len(queue), f"({len(queue)/len(baseline):.1%} of all pages)")
print("Reason code on every row:", queue['reason_code'].unique())
print()
print("Human review, always: audience quality (bot-inflation check), business/editorial")
print("context, whether the recorded update date reflects a real edit.")
print()
print("Never automate from the score alone: no auto-publish, auto-rewrite, or auto-prune;")
print("never treat MONITOR as 'confirmed healthy' for pages with missing staleness data;")
print("never present the score as a causal forecast.")
print()
print("Monitoring triggers: shifts in the staleness-observed rate (currently "
      f"{staleness_rate:.1%}), in the share crossing the score-70 cut (currently "
      f"{(baseline['baseline_score']>=70).mean():.1%}), or in the zero-impressions share "
      f"(currently {(baseline['march_impressions']==0).mean():.1%})."
)


Review queue size: 10189 (3.1% of all pages)
Reason code on every row: <StringArray>
['STALE_HIGH_OPPORTUNITY']
Length: 1, dtype: str

Human review, always: audience quality (bot-inflation check), business/editorial
context, whether the recorded update date reflects a real edit.

Never automate from the score alone: no auto-publish, auto-rewrite, or auto-prune;
never treat MONITOR as 'confirmed healthy' for pages with missing staleness data;
never present the score as a causal forecast.

Monitoring triggers: shifts in the staleness-observed rate (currently 11.5%), in the share crossing the score-70 cut (currently 3.1%), or in the zero-impressions share (currently 46.7%).


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

os.makedirs("work/outputs/figures", exist_ok=True)

# Figure 1: baseline vs model Precision@50
fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(["Baseline", "Model"], [baseline_p50, model_p50], color=["#3E5C63", "#A8632B"])
ax.set_xlim(0, 1)
ax.set_xlabel("Precision@50")
ax.set_title("Baseline vs. model, single split")
for i, v in enumerate([baseline_p50, model_p50]):
    ax.text(v + 0.02, i, f"{v:.2f}", va="center")
fig.tight_layout()
fig.savefig("work/outputs/figures/baseline_vs_model.png", dpi=150)
plt.close(fig)

# Figure 2: 5-fold CV spread
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(fold_scores, [0]*len(fold_scores), color="#3E5C63", zorder=3, label="folds")
ax.scatter([model_p50], [0], color="#A8632B", zorder=4, label="single split (0.66)")
ax.axvline(fold_scores.mean(), color="gray", linestyle="--", label=f"mean {fold_scores.mean():.3f}")
ax.set_yticks([])
ax.set_xlabel("Precision@50")
ax.set_title("Model stability: single split vs. 5-fold CV")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("work/outputs/figures/cv_stability.png", dpi=150)
plt.close(fig)

# Figure 3: feature importances
fig, ax = plt.subplots(figsize=(5, 3))
importances.sort_values().plot(kind="barh", ax=ax, color="#A8632B")
ax.set_xlabel("Importance")
ax.set_title("What the model leans on")
fig.tight_layout()
fig.savefig("work/outputs/figures/feature_importances.png", dpi=150)
plt.close(fig)

print("Saved figures:")
for f in sorted(os.listdir("work/outputs/figures")):
    print(" -", f)
print()
print("Tables embedded in the paper: baseline-vs-model Precision@50 table (Results),")
print("notebook-to-paper-section map (Reproducibility).")


Saved figures:
 - baseline_vs_model.png
 - cv_stability.png
 - feature_importances.png

Tables embedded in the paper: baseline-vs-model Precision@50 table (Results),
notebook-to-paper-section map (Reproducibility).


## ML-12 — Demo, social post, employer summary

Closing artifacts for sharing the work outside the notebook.

In [ ]:
demo_outline = '''
5-minute demo outline
1. (30s) The problem: a content team can't manually review every page -- which ones go first?
2. (60s) The baseline rule: two signals, one threshold, 10,189 pages flagged REVIEW_REFRESH.
3. (90s) The model: five features, Random Forest, Precision@50 0.42 -> 0.66 on the same
   held-out pages; show the bar chart.
4. (60s) The audit, honestly: cross-validation shows 0.73 +/- 0.05, not a fixed 0.66;
   zero-impression pages can never be labeled declining, so part of the lift is structural,
   not learned. Staleness data only exists for 11.5% of pages.
5. (60s) The playbook: ranked queue, human-review checklist, no-go list, monitoring triggers.
'''

social_post = (
    "Built a content-refresh priority model on 331K anonymized pages from the FlyRank "
    "ML Internship dataset. Baseline rule -> Precision@50 of 0.42. A 5-feature Random "
    "Forest lifts that to 0.66 (0.73 +/- 0.05 cross-validated) -- but the honest finding "
    "is structural: pages with zero traffic can never be labeled 'declining', so part of "
    "the gain is an easy floor, not fine-grained skill. Full paper + audit: [repo link]."
)

employer_summary = (
    "Built and validated a ranking model that prioritizes which of 331,437 content pages "
    "a review team should look at first, improving Precision@50 from a 0.42 baseline to "
    "0.66 (0.73 +/- 0.05 under 5-fold cross-validation), while running two independent "
    "leakage audits that surfaced a structural artifact in the label definition -- "
    "delivered as a public research paper with full reproducibility."
)

print(demo_outline)
print("SOCIAL POST:\n", social_post, "\n")
print("EMPLOYER SUMMARY:\n", employer_summary)



5-minute demo outline
1. (30s) The problem: a content team can't manually review every page -- which ones go first?
2. (60s) The baseline rule: two signals, one threshold, 10,189 pages flagged REVIEW_REFRESH.
3. (90s) The model: five features, Random Forest, Precision@50 0.42 -> 0.66 on the same
   held-out pages; show the bar chart.
4. (60s) The audit, honestly: cross-validation shows 0.73 +/- 0.05, not a fixed 0.66;
   zero-impression pages can never be labeled declining, so part of the lift is structural,
   not learned. Staleness data only exists for 11.5% of pages.
5. (60s) The playbook: ranked queue, human-review checklist, no-go list, monitoring triggers.

SOCIAL POST:
 Built a content-refresh priority model on 331K anonymized pages from the FlyRank ML Internship dataset. Baseline rule -> Precision@50 of 0.42. A 5-feature Random Forest lifts that to 0.66 (0.73 +/- 0.05 cross-validated) -- but the honest finding is structural: pages with zero traffic can never be labeled 'declin

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
